# Linear Model - MNIST

In [ ]:
%pip install --quiet mlflow==3.6.0 scikit-learn==1.8.0
dbutils.library.restartPython()

In [ ]:
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import logging
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Get volume_path from DAB parameter (or use default)
seed = int(dbutils.widgets.get('random_state'))
experiment_name = dbutils.widgets.get('experiment_name')
try:
    volume_path = dbutils.widgets.get('volume_path')
except:
    volume_path = '/Volumes/ai_ml_in_practice/mnist_data/processed'

print(f'Volume path: {volume_path}\n')

# Load MNIST data from volume
x_train = np.load(os.path.join(volume_path, 'x_train.npy'))
y_train = np.load(os.path.join(volume_path, 'y_train.npy'))
x_val = np.load(os.path.join(volume_path, 'x_val.npy'))
y_val = np.load(os.path.join(volume_path, 'y_val.npy'))

print(f'Loaded - train: {x_train.shape}, val: {x_val.shape}')

mlflow.set_registry_uri('databricks-uc')
mlflow.set_experiment(experiment_name)

In [ ]:
# Train Logistic Regression
with mlflow.start_run(run_name='logistic_mnist') as run:
    mlflow.log_param('model', 'logistic_regression')
    mlflow.log_param('max_iter', 1000)
    
    model = LogisticRegression(max_iter=1000, random_state=seed)
    model.fit(x_train, y_train)
    run_id = run.info.run_id
    
    y_pred = model.predict(x_val)
    y_proba = model.predict_proba(x_val)
    
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_val, y_pred, average='weighted', zero_division=0)
    auc = roc_auc_score(y_val, y_proba, multi_class='ovr', average='weighted')
    
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('precision', prec)
    mlflow.log_metric('recall', rec)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('auc', auc)
    
    signature = infer_signature(x_val, y_pred)
    mlflow.sklearn.log_model(model, 'model', signature=signature)
    mlflow.set_tag('dataset', 'mnist')
    
    print(f'Logistic Regression - Accuracy: {acc:.4f}, F1: {f1:.4f}')

# Pass run_id to next task
try:
    dbutils.jobs.taskValues.set(key='linear_run_id', value=run_id)
except NameError:
    pass